# Search Greedy


In [2]:
"""
==============================================================================
MODULO DI RICERCA NELLO SPAZIO DEGLI STATI
Algoritmo: Greedy Best-First Search (GBFS)
Applicazione: Problema del Commesso Viaggiatore (TSP) per routing consegne
==============================================================================

TEORIA (Capitolo 2.2.3 delle dispense):
- GBFS è un algoritmo di ricerca informata che utilizza una funzione euristica h(n)
- Espande sempre il nodo che APPARE più vicino all'obiettivo secondo h(n)
- Euristica usata: distanza in linea d'aria (Haversine) verso il nodo più vicino
- Complessità: O(b^m) nel caso peggiore, ma spesso molto efficiente in pratica
- NON garantisce la soluzione ottimale (è un algoritmo greedy)
"""

import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2
from typing import List, Tuple, Optional
from dataclasses import dataclass
import folium


# ==============================================================================
# SEZIONE 1: STRUTTURE DATI
# ==============================================================================

@dataclass
class DeliveryNode:
    """
    Rappresenta un nodo nel grafo delle consegne.
    Ogni nodo corrisponde a un punto di consegna o al deposito.
    """
    node_id: int
    latitude: float
    longitude: float
    package_id: Optional[str] = None
    is_depot: bool = False
    
    def __repr__(self):
        if self.is_depot:
            return f"Deposito({self.latitude:.4f}, {self.longitude:.4f})"
        return f"Nodo_{self.node_id}(pkg={self.package_id})"


# ==============================================================================
# SEZIONE 2: FUNZIONI EURISTICHE
# ==============================================================================

def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Calcola la distanza in linea d'aria tra due punti geografici.
    
    EURISTICA h(n): Questa funzione implementa l'euristica del GBFS.
    Usa la formula di Haversine che tiene conto della curvatura terrestre.
    
    Args:
        lat1, lon1: Coordinate del primo punto (gradi decimali)
        lat2, lon2: Coordinate del secondo punto (gradi decimali)
    
    Returns:
        Distanza in chilometri
    
    Nota: L'euristica è AMMISSIBILE per il TSP locale perché la distanza
    in linea d'aria non sovrastima mai la distanza reale.
    """
    # Raggio medio della Terra in km
    R = 6371.0
    
    # Conversione da gradi a radianti
    lat1_rad = radians(lat1)
    lat2_rad = radians(lat2)
    delta_lat = radians(lat2 - lat1)
    delta_lon = radians(lon2 - lon1)
    
    # Formula di Haversine
    a = sin(delta_lat / 2)**2 + cos(lat1_rad) * cos(lat2_rad) * sin(delta_lon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    
    distance = R * c
    return distance


# ==============================================================================
# SEZIONE 3: ALGORITMO GREEDY BEST-FIRST SEARCH
# ==============================================================================

class GreedyRouteSolver:
    """
    Implementazione del GBFS per il problema TSP delle consegne.
    
    ALGORITMO GBFS (Greedy Best-First Search):
    1. Inizializza: current_node = deposito, unvisited = tutti i nodi
    2. Mentre ci sono nodi non visitati:
       a. Calcola h(n) = distanza Haversine per ogni nodo non visitato
       b. Seleziona il nodo con h(n) MINIMA (strategia greedy)
       c. Sposta al nodo selezionato
       d. Rimuovi il nodo dalla lista dei non visitati
    3. Ritorna al deposito
    
    Differenza con A*: GBFS usa SOLO h(n), non f(n) = g(n) + h(n)
    Questo lo rende più veloce ma NON ottimale.
    """
    
    def __init__(self, depot: DeliveryNode, delivery_nodes: List[DeliveryNode]):
        """
        Inizializza il solver con il deposito e i nodi da visitare.
        
        Args:
            depot: Nodo deposito (Start/End)
            delivery_nodes: Lista dei nodi di consegna
        """
        self.depot = depot
        self.delivery_nodes = delivery_nodes.copy()
        
        # Stato della ricerca
        self.route: List[DeliveryNode] = []  # Percorso trovato
        self.total_distance: float = 0.0      # Distanza totale
        self.search_steps: List[dict] = []    # Log dei passi per debug
        
    def _find_nearest_unvisited(self, 
                                 current: DeliveryNode, 
                                 unvisited: List[DeliveryNode]) -> Tuple[DeliveryNode, float]:
        """
        CUORE DEL GBFS: Trova il nodo non visitato con euristica minima.
        
        Questa funzione implementa la strategia greedy:
        - Calcola h(n) per ogni nodo non visitato
        - Restituisce quello con valore MINIMO
        
        Args:
            current: Nodo corrente
            unvisited: Lista dei nodi ancora da visitare
            
        Returns:
            Tupla (nodo_più_vicino, distanza)
        """
        best_node = None
        best_distance = float('inf')
        
        # Valuta tutti i nodi non visitati
        for node in unvisited:
            # Calcolo dell'euristica h(n) = distanza Haversine
            h_n = haversine_distance(
                current.latitude, current.longitude,
                node.latitude, node.longitude
            )
            
            # Strategia Greedy: scegli il minimo
            if h_n < best_distance:
                best_distance = h_n
                best_node = node
                
        return best_node, best_distance
    
    def solve(self) -> List[DeliveryNode]:
        """
        Esegue l'algoritmo GBFS per trovare il percorso.
        
        PSEUDOCODICE GBFS per TSP:
        ─────────────────────────
        function GBFS_TSP(deposito, nodi):
            percorso ← [deposito]
            non_visitati ← nodi
            corrente ← deposito
            
            while non_visitati non è vuoto:
                # Espansione: trova il nodo con h(n) minima
                prossimo ← argmin{h(corrente, n) : n ∈ non_visitati}
                
                # Aggiorna lo stato
                percorso.append(prossimo)
                non_visitati.remove(prossimo)
                corrente ← prossimo
            
            # Ritorno al deposito
            percorso.append(deposito)
            return percorso
        
        Returns:
            Lista ordinata dei nodi nel percorso ottimizzato
        """
        print("\n" + "="*60)
        print("ESECUZIONE GREEDY BEST-FIRST SEARCH (GBFS)")
        print("="*60)
        
        # Inizializzazione
        self.route = [self.depot]
        self.total_distance = 0.0
        unvisited = self.delivery_nodes.copy()
        current = self.depot
        step = 0
        
        print(f"\n📍 Partenza dal Deposito: ({self.depot.latitude:.4f}, {self.depot.longitude:.4f})")
        print(f"📦 Nodi da visitare: {len(unvisited)}")
        print("\n" + "-"*60)
        
        # Loop principale GBFS
        while unvisited:
            step += 1
            
            # PASSO CHIAVE: Trova il nodo con h(n) minima
            next_node, distance = self._find_nearest_unvisited(current, unvisited)
            
            # Log del passo di ricerca
            self.search_steps.append({
                'step': step,
                'from': current,
                'to': next_node,
                'h_n': distance,
                'remaining': len(unvisited) - 1
            })
            
            print(f"Passo {step:2d}: {current} → {next_node}")
            print(f"         h(n) = {distance:.2f} km | Rimanenti: {len(unvisited)-1}")
            
            # Aggiornamento stato
            self.route.append(next_node)
            self.total_distance += distance
            unvisited.remove(next_node)
            current = next_node
        
        # Ritorno al deposito (chiusura del ciclo)
        return_distance = haversine_distance(
            current.latitude, current.longitude,
            self.depot.latitude, self.depot.longitude
        )
        self.route.append(self.depot)
        self.total_distance += return_distance
        
        print("-"*60)
        print(f"Passo {step+1:2d}: {current} → {self.depot} (Ritorno)")
        print(f"         Distanza ritorno: {return_distance:.2f} km")
        print("="*60)
        print(f"\n✅ PERCORSO COMPLETATO!")
        print(f"📏 DISTANZA TOTALE: {self.total_distance:.2f} km")
        print("="*60)
        
        return self.route
    
    def get_route_dataframe(self) -> pd.DataFrame:
        """
        Crea un DataFrame con il percorso ordinato per l'export.
        """
        route_data = []
        for i, node in enumerate(self.route):
            route_data.append({
                'Order': i,
                'Node_Type': 'Depot' if node.is_depot else 'Delivery',
                'Package_ID': node.package_id if not node.is_depot else 'DEPOT',
                'Latitude': node.latitude,
                'Longitude': node.longitude
            })
        return pd.DataFrame(route_data)


# ==============================================================================
# SEZIONE 4: VISUALIZZAZIONE CON FOLIUM
# ==============================================================================

class RouteVisualizer:
    """
    Crea una mappa interattiva del percorso usando Folium.
    """
    
    def __init__(self, route: List[DeliveryNode], total_distance: float):
        self.route = route
        self.total_distance = total_distance
        
    def create_map(self, output_path: str = "delivery_route_map.html") -> folium.Map:
        """
        Genera la mappa HTML con il percorso visualizzato.
        
        Args:
            output_path: Percorso del file HTML di output
            
        Returns:
            Oggetto folium.Map
        """
        # Calcola il centro della mappa
        lats = [node.latitude for node in self.route]
        lons = [node.longitude for node in self.route]
        center_lat = np.mean(lats)
        center_lon = np.mean(lons)
        
        # Crea la mappa
        route_map = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=12,
            tiles='OpenStreetMap'
        )
        
        # Aggiungi titolo
        title_html = f'''
        <div style="position: fixed; 
                    top: 10px; left: 50px; width: 400px;
                    background-color: white; padding: 10px;
                    border: 2px solid grey; border-radius: 5px;
                    z-index: 9999; font-family: Arial;">
            <h4>🚚 Percorso Furgone 0 - GBFS</h4>
            <p>Distanza Totale: <b>{self.total_distance:.2f} km</b></p>
            <p>Consegne: <b>{len(self.route) - 2}</b></p>
        </div>
        '''
        route_map.get_root().html.add_child(folium.Element(title_html))
        
        # Aggiungi marker per il Deposito (Rosso)
        depot = self.route[0]
        folium.Marker(
            location=[depot.latitude, depot.longitude],
            popup="🏭 DEPOSITO (Start/End)",
            icon=folium.Icon(color='red', icon='home', prefix='fa'),
            tooltip="Deposito"
        ).add_to(route_map)
        
        # Aggiungi marker numerati per le consegne (Blu)
        delivery_order = 1
        for node in self.route[1:-1]:  # Escludi deposito iniziale e finale
            folium.Marker(
                location=[node.latitude, node.longitude],
                popup=f"📦 Consegna #{delivery_order}<br>Package: {node.package_id}",
                icon=folium.DivIcon(
                    html=f'''
                    <div style="
                        background-color: #3388ff;
                        color: white;
                        border-radius: 50%;
                        width: 25px;
                        height: 25px;
                        text-align: center;
                        line-height: 25px;
                        font-weight: bold;
                        font-size: 12px;
                        border: 2px solid white;
                        box-shadow: 2px 2px 5px rgba(0,0,0,0.3);
                    ">{delivery_order}</div>
                    ''',
                    icon_size=(25, 25),
                    icon_anchor=(12, 12)
                ),
                tooltip=f"Consegna #{delivery_order}"
            ).add_to(route_map)
            delivery_order += 1
        
        # Disegna la PolyLine del percorso
        route_coords = [[node.latitude, node.longitude] for node in self.route]
        folium.PolyLine(
            locations=route_coords,
            weight=3,
            color='blue',
            opacity=0.8,
            dash_array='10',
            tooltip="Percorso GBFS"
        ).add_to(route_map)
        
        # Aggiungi frecce direzionali
        from folium.plugins import AntPath
        AntPath(
            locations=route_coords,
            weight=2,
            color='#0066cc',
            pulse_color='#ffffff',
            delay=1000
        ).add_to(route_map)
        
        # Salva la mappa
        route_map.save(output_path)
        print(f"\n🗺️  Mappa salvata in: {output_path}")
        
        return route_map


# ==============================================================================
# SEZIONE 5: MAIN - CARICAMENTO DATI E ESECUZIONE
# ==============================================================================

def main():
    """
    Funzione principale che orchestra l'intero processo:
    1. Caricamento e preprocessing dei dati
    2. Creazione del grafo delle consegne
    3. Esecuzione GBFS
    4. Visualizzazione e export
    """
    print("\n" + "="*60)
    print("  MODULO RICERCA: TSP con Greedy Best-First Search")
    print("  Corso: Intelligenza Artificiale")
    print("="*60)
    
    # ─────────────────────────────────────────────────────────────
    # STEP 1: Caricamento dei dati
    # ─────────────────────────────────────────────────────────────
    print("\n📂 STEP 1: Caricamento dati...")
    
    try:
        # Carica il CSV ottimizzato dal modulo CSP
        df = pd.read_csv('../CSP/amazon_delivery_optimized.csv')
        print(f"   Dataset caricato: {len(df)} righe totali")
    except FileNotFoundError:
        print("   ⚠️ File non trovato nel percorso CSP, provo percorso alternativo...")
        try:
            df = pd.read_csv('amazon_delivery_optimized.csv')
            print(f"   Dataset caricato: {len(df)} righe totali")
        except FileNotFoundError:
            print("   ❌ Errore: Impossibile trovare amazon_delivery_optimized.csv")
            print("   Assicurati di aver eseguito prima il modulo CSP.")
            return
    
    # ─────────────────────────────────────────────────────────────
    # STEP 2: Filtraggio per Furgone 0
    # ─────────────────────────────────────────────────────────────
    print("\n🔍 STEP 2: Filtraggio dati Furgone 0...")
    
    # Filtra solo le righe del Furgone 0 con stato 'Loaded'
    van_0_df = df[(df['Van_ID'] == 0) & (df['Status'] == 'Loaded')].copy()
    
    if len(van_0_df) == 0:
        # Fallback: prova solo con Van_ID
        van_0_df = df[df['Van_ID'] == 0].copy()
        print(f"   ⚠️ Nessun pacco 'Loaded', uso tutti i pacchi del Van 0")
    
    print(f"   Pacchi assegnati al Furgone 0: {len(van_0_df)}")
    
    if len(van_0_df) == 0:
        print("   ❌ Nessun pacco trovato per il Furgone 0!")
        return
    
    # ─────────────────────────────────────────────────────────────
    # STEP 3: Creazione del Deposito
    # ─────────────────────────────────────────────────────────────
    print("\n🏭 STEP 3: Calcolo posizione Deposito...")
    
    # Il deposito è la media delle coordinate del negozio
    depot_lat = van_0_df['Store_Latitude'].mean()
    depot_lon = van_0_df['Store_Longitude'].mean()
    
    depot = DeliveryNode(
        node_id=0,
        latitude=depot_lat,
        longitude=depot_lon,
        is_depot=True
    )
    print(f"   Deposito: ({depot_lat:.4f}, {depot_lon:.4f})")
    
    # ─────────────────────────────────────────────────────────────
    # STEP 4: Creazione dei nodi di consegna
    # ─────────────────────────────────────────────────────────────
    print("\n📦 STEP 4: Creazione nodi di consegna...")
    
    delivery_nodes = []
    for idx, row in van_0_df.iterrows():
        node = DeliveryNode(
            node_id=len(delivery_nodes) + 1,
            latitude=row['Drop_Latitude'],
            longitude=row['Drop_Longitude'],
            package_id=str(row.get('ID', idx))
        )
        delivery_nodes.append(node)
    
    print(f"   Creati {len(delivery_nodes)} nodi di consegna")
    
    # ─────────────────────────────────────────────────────────────
    # STEP 5: Esecuzione GBFS
    # ─────────────────────────────────────────────────────────────
    print("\n🔄 STEP 5: Esecuzione algoritmo GBFS...")
    
    solver = GreedyRouteSolver(depot, delivery_nodes)
    optimal_route = solver.solve()
    
    # ─────────────────────────────────────────────────────────────
    # STEP 6: Visualizzazione
    # ─────────────────────────────────────────────────────────────
    print("\n🗺️  STEP 6: Generazione mappa interattiva...")
    
    visualizer = RouteVisualizer(optimal_route, solver.total_distance)
    visualizer.create_map("delivery_route_map.html")
    
    # ─────────────────────────────────────────────────────────────
    # STEP 7: Export CSV
    # ─────────────────────────────────────────────────────────────
    print("\n💾 STEP 7: Export percorso in CSV...")
    
    route_df = solver.get_route_dataframe()
    route_df.to_csv("route_van_0.csv", index=False)
    print(f"   Percorso salvato in: route_van_0.csv")
    
    # ─────────────────────────────────────────────────────────────
    # RIEPILOGO FINALE
    # ─────────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("  RIEPILOGO ESECUZIONE")
    print("="*60)
    print(f"  • Algoritmo utilizzato: Greedy Best-First Search (GBFS)")
    print(f"  • Euristica h(n): Distanza Haversine (linea d'aria)")
    print(f"  • Nodi visitati: {len(delivery_nodes)}")
    print(f"  • Distanza totale: {solver.total_distance:.2f} km")
    print(f"  • File generati:")
    print(f"    - delivery_route_map.html (mappa interattiva)")
    print(f"    - route_van_0.csv (percorso ordinato)")
    print("="*60)
    
    return solver, optimal_route


# ==============================================================================
# ESECUZIONE
# ==============================================================================

if __name__ == "__main__":
    solver, route = main()

    # Aggiungi questa nuova cella dopo l'esecuzione del main()

def create_enhanced_map(route: list, total_distance: float, output_path: str = "delivery_route_enhanced.html"):
    """
    Crea una mappa migliorata simile all'immagine di riferimento.
    Utilizza OpenStreetMap con uno stile più pulito e linee più visibili.
    """
    import folium
    from folium.plugins import AntPath
    import numpy as np
    
    # Calcola il centro della mappa
    lats = [node.latitude for node in route]
    lons = [node.longitude for node in route]
    center_lat = np.mean(lats)
    center_lon = np.mean(lons)
    
    # Calcola i bounds per lo zoom automatico
    min_lat, max_lat = min(lats), max(lats)
    min_lon, max_lon = min(lons), max(lons)
    
    # Crea la mappa con tiles OpenStreetMap standard
    route_map = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=12,
        tiles='OpenStreetMap',
        control_scale=True
    )
    
    # Fit ai bounds del percorso
    route_map.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]], padding=[20, 20])
    
    # ══════════════════════════════════════════════════════════════
    # MARKER DEPOSITO (Rosso con icona casa)
    # ══════════════════════════════════════════════════════════════
    depot = route[0]
    
    # Marker rosso per il deposito (Start)
    folium.Marker(
        location=[depot.latitude, depot.longitude],
        popup=folium.Popup(
            f"""
            <div style="font-family: Arial; min-width: 150px;">
                <h4 style="color: #d63031; margin: 5px 0;">🏭 DEPOSITO</h4>
                <hr style="margin: 5px 0;">
                <p><b>Tipo:</b> Start/End Point</p>
                <p><b>Lat:</b> {depot.latitude:.6f}</p>
                <p><b>Lon:</b> {depot.longitude:.6f}</p>
            </div>
            """,
            max_width=250
        ),
        icon=folium.Icon(
            color='red',
            icon='home',
            prefix='fa'
        ),
        tooltip="🏭 Deposito (Partenza/Arrivo)"
    ).add_to(route_map)
    
    # ══════════════════════════════════════════════════════════════
    # MARKER CONSEGNE (Verde con numeri)
    # ══════════════════════════════════════════════════════════════
    delivery_order = 1
    for node in route[1:-1]:  # Escludi deposito iniziale e finale
        # Marker verde con numero
        folium.Marker(
            location=[node.latitude, node.longitude],
            popup=folium.Popup(
                f"""
                <div style="font-family: Arial; min-width: 180px;">
                    <h4 style="color: #00b894; margin: 5px 0;">📦 Consegna #{delivery_order}</h4>
                    <hr style="margin: 5px 0;">
                    <p><b>Package ID:</b> {node.package_id}</p>
                    <p><b>Lat:</b> {node.latitude:.6f}</p>
                    <p><b>Lon:</b> {node.longitude:.6f}</p>
                </div>
                """,
                max_width=250
            ),
            icon=folium.DivIcon(
                html=f'''
                <div style="
                    background-color: #00b894;
                    color: white;
                    border-radius: 50%;
                    width: 28px;
                    height: 28px;
                    text-align: center;
                    line-height: 28px;
                    font-weight: bold;
                    font-size: 11px;
                    border: 3px solid white;
                    box-shadow: 0 2px 6px rgba(0,0,0,0.4);
                ">{delivery_order}</div>
                ''',
                icon_size=(28, 28),
                icon_anchor=(14, 14)
            ),
            tooltip=f"📦 Consegna #{delivery_order} - {node.package_id}"
        ).add_to(route_map)
        delivery_order += 1
    
    # ══════════════════════════════════════════════════════════════
    # PERCORSO - Linea Blu Spessa (come nell'immagine)
    # ══════════════════════════════════════════════════════════════
    route_coords = [[node.latitude, node.longitude] for node in route]
    
    # Linea principale del percorso (blu scuro, spessa)
    folium.PolyLine(
        locations=route_coords,
        weight=4,           # Spessore maggiore
        color='#0066cc',    # Blu scuro
        opacity=0.9,
        tooltip=f"Percorso GBFS - {total_distance:.2f} km"
    ).add_to(route_map)
    
    # Linea animata sopra (effetto "formiche in marcia")
    AntPath(
        locations=route_coords,
        weight=3,
        color='#0066cc',
        pulse_color='#ffffff',
        delay=800,
        dash_array=[10, 20]
    ).add_to(route_map)
    
    # ══════════════════════════════════════════════════════════════
    # LEGENDA E INFO BOX
    # ══════════════════════════════════════════════════════════════
    legend_html = f'''
    <div style="
        position: fixed; 
        bottom: 30px; 
        left: 30px; 
        width: 280px;
        background-color: white; 
        padding: 15px;
        border: 2px solid #333; 
        border-radius: 8px;
        z-index: 9999; 
        font-family: Arial, sans-serif;
        box-shadow: 0 4px 12px rgba(0,0,0,0.3);
    ">
        <h4 style="margin: 0 0 10px 0; color: #2d3436; border-bottom: 2px solid #0066cc; padding-bottom: 8px;">
            🚚 Percorso Furgone 0
        </h4>
        <p style="margin: 5px 0; font-size: 14px;">
            <b>Algoritmo:</b> Greedy Best-First Search
        </p>
        <p style="margin: 5px 0; font-size: 14px;">
            <b>Distanza Totale:</b> <span style="color: #d63031; font-weight: bold;">{total_distance:.2f} km</span>
        </p>
        <p style="margin: 5px 0; font-size: 14px;">
            <b>N° Consegne:</b> {len(route) - 2}
        </p>
        <hr style="margin: 10px 0; border-color: #ddd;">
        <div style="font-size: 12px;">
            <p style="margin: 3px 0;">
                <span style="color: #d63031;">●</span> Deposito (Start/End)
            </p>
            <p style="margin: 3px 0;">
                <span style="color: #00b894;">●</span> Punti di Consegna
            </p>
            <p style="margin: 3px 0;">
                <span style="color: #0066cc;">━━</span> Percorso Ottimizzato
            </p>
        </div>
    </div>
    '''
    route_map.get_root().html.add_child(folium.Element(legend_html))
    
    # ══════════════════════════════════════════════════════════════
    # TITOLO IN ALTO
    # ══════════════════════════════════════════════════════════════
    title_html = '''
    <div style="
        position: fixed; 
        top: 10px; 
        left: 50%; 
        transform: translateX(-50%);
        background-color: rgba(255,255,255,0.95); 
        padding: 10px 25px;
        border: 2px solid #0066cc; 
        border-radius: 25px;
        z-index: 9999; 
        font-family: Arial, sans-serif;
        box-shadow: 0 3px 10px rgba(0,0,0,0.2);
    ">
        <h3 style="margin: 0; color: #2d3436; font-size: 16px;">
            🗺️ TSP Routing - Greedy Best-First Search (GBFS)
        </h3>
    </div>
    '''
    route_map.get_root().html.add_child(folium.Element(title_html))
    
    # Salva la mappa
    route_map.save(output_path)
    print(f"\n✅ Mappa migliorata salvata in: {output_path}")
    
    return route_map


# ══════════════════════════════════════════════════════════════
# ESECUZIONE - Genera la nuova mappa
# ══════════════════════════════════════════════════════════════
if 'solver' in dir() and 'route' in dir():
    enhanced_map = create_enhanced_map(route, solver.total_distance, "delivery_route_enhanced.html")
    print("\n🎉 Apri 'delivery_route_enhanced.html' nel browser per visualizzare il percorso!")
else:
    print("⚠️ Esegui prima la cella con main() per generare solver e route")


  MODULO RICERCA: TSP con Greedy Best-First Search
  Corso: Intelligenza Artificiale

📂 STEP 1: Caricamento dati...
   Dataset caricato: 39997 righe totali

🔍 STEP 2: Filtraggio dati Furgone 0...
   Pacchi assegnati al Furgone 0: 39

🏭 STEP 3: Calcolo posizione Deposito...
   Deposito: (23.5586, 85.2570)

📦 STEP 4: Creazione nodi di consegna...
   Creati 39 nodi di consegna

🔄 STEP 5: Esecuzione algoritmo GBFS...

ESECUZIONE GREEDY BEST-FIRST SEARCH (GBFS)

📍 Partenza dal Deposito: (23.5586, 85.2570)
📦 Nodi da visitare: 39

------------------------------------------------------------
Passo  1: Deposito(23.5586, 85.2570) → Nodo_7(pkg=3591)
         h(n) = 15.45 km | Rimanenti: 38
Passo  2: Nodo_7(pkg=3591) → Nodo_20(pkg=3604)
         h(n) = 0.00 km | Rimanenti: 37
Passo  3: Nodo_20(pkg=3604) → Nodo_4(pkg=3588)
         h(n) = 4.66 km | Rimanenti: 36
Passo  4: Nodo_4(pkg=3588) → Nodo_27(pkg=3611)
         h(n) = 0.00 km | Rimanenti: 35
Passo  5: Nodo_27(pkg=3611) → Nodo_11(pkg=3595)
  

## Branch and Bound


In [6]:
"""
==============================================================================
BENCHMARK: Greedy Best-First Search vs Branch and Bound
==============================================================================
Corso: Intelligenza Artificiale - Ricerca Operativa
Riferimenti Teorici:
  - Capitolo 2.3.1: Branch and Bound
  - Capitolo 2.2.6: Potatura (Pruning)

DESCRIZIONE:
Questo script confronta due algoritmi per il Problema del Commesso Viaggiatore (TSP):
1. GBFS (Greedy Best-First Search): Algoritmo greedy, veloce ma sub-ottimale
2. B&B (Branch and Bound): Algoritmo esatto, ottimale ma con complessità esponenziale

STRATEGIA B&B:
- Esplora l'albero delle permutazioni in modo sistematico
- Usa una variabile "Upper Bound" (UB) inizializzata con il costo del Greedy
- POTATURA: Se il costo parziale >= UB, il ramo viene tagliato (pruning)
- Quando trova una soluzione completa migliore, aggiorna UB
==============================================================================
"""

import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2, factorial
from typing import List, Tuple, Optional
from dataclasses import dataclass
import time
from tabulate import tabulate

# ==============================================================================
# SEZIONE 1: STRUTTURE DATI E FUNZIONI DI UTILITÀ
# ==============================================================================

@dataclass
class Node:
    """Rappresenta un nodo (punto di consegna o deposito)."""
    id: int
    latitude: float
    longitude: float
    name: str = ""
    
    def __repr__(self):
        return f"Node({self.id}: {self.name})"


def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Calcola la distanza in km tra due punti usando la formula di Haversine.
    Tiene conto della curvatura terrestre.
    """
    R = 6371.0  # Raggio della Terra in km
    
    lat1_rad, lat2_rad = radians(lat1), radians(lat2)
    delta_lat = radians(lat2 - lat1)
    delta_lon = radians(lon2 - lon1)
    
    a = sin(delta_lat / 2)**2 + cos(lat1_rad) * cos(lat2_rad) * sin(delta_lon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    
    return R * c


def build_distance_matrix(nodes: List[Node]) -> np.ndarray:
    """
    Pre-calcola la matrice NxN delle distanze Haversine.
    
    OTTIMIZZAZIONE: Evita di ricalcolare le distanze durante la ricorsione B&B.
    La matrice è simmetrica: dist[i][j] = dist[j][i]
    
    Args:
        nodes: Lista di nodi (indice 0 = deposito)
    
    Returns:
        Matrice numpy NxN con le distanze in km
    """
    n = len(nodes)
    dist_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i + 1, n):
            d = haversine_distance(
                nodes[i].latitude, nodes[i].longitude,
                nodes[j].latitude, nodes[j].longitude
            )
            dist_matrix[i][j] = d
            dist_matrix[j][i] = d  # Simmetria
    
    return dist_matrix


# ==============================================================================
# SEZIONE 2: ALGORITMO GREEDY BEST-FIRST SEARCH (BASELINE)
# ==============================================================================

def greedy_tsp(dist_matrix: np.ndarray, start: int = 0) -> Tuple[List[int], float]:
    """
    Algoritmo Greedy per TSP: ad ogni passo sceglie il nodo più vicino.
    
    STRATEGIA GREEDY:
    - Parte dal deposito (nodo 0)
    - Ad ogni iterazione, sceglie il nodo non visitato con distanza minima
    - Ritorna al deposito alla fine
    
    COMPLESSITÀ: O(N²) - efficiente ma NON ottimale
    
    Args:
        dist_matrix: Matrice delle distanze pre-calcolata
        start: Indice del nodo di partenza (default: 0 = deposito)
    
    Returns:
        Tupla (percorso, costo_totale)
    """
    n = len(dist_matrix)
    visited = [False] * n
    route = [start]
    visited[start] = True
    total_cost = 0.0
    current = start
    
    # Visita tutti i nodi scegliendo sempre il più vicino
    for _ in range(n - 1):
        best_next = -1
        best_dist = float('inf')
        
        for j in range(n):
            if not visited[j] and dist_matrix[current][j] < best_dist:
                best_dist = dist_matrix[current][j]
                best_next = j
        
        if best_next != -1:
            route.append(best_next)
            visited[best_next] = True
            total_cost += best_dist
            current = best_next
    
    # Ritorno al deposito
    total_cost += dist_matrix[current][start]
    route.append(start)
    
    return route, total_cost


# ==============================================================================
# SEZIONE 3: ALGORITMO BRANCH AND BOUND (CHALLENGER)
# ==============================================================================

class BranchAndBoundTSP:
    """
    Implementazione Branch and Bound per TSP.
    
    TEORIA (Capitolo 2.3.1):
    - L'algoritmo esplora l'albero delle permutazioni
    - Ogni nodo dell'albero rappresenta un percorso parziale
    - Si usa un Upper Bound (UB) per potare rami non promettenti
    
    COMPONENTI CHIAVE:
    1. BRANCHING: Espansione dei nodi figli (tutti i nodi non ancora visitati)
    2. BOUNDING: Calcolo del costo parziale
    3. PRUNING: Taglio dei rami se costo_parziale >= Upper Bound
    """
    
    def __init__(self, dist_matrix: np.ndarray, initial_upper_bound: float = float('inf')):
        """
        Inizializza il solver B&B.
        
        Args:
            dist_matrix: Matrice delle distanze
            initial_upper_bound: UB iniziale (usa il risultato del Greedy!)
        """
        self.dist_matrix = dist_matrix
        self.n = len(dist_matrix)
        
        # ══════════════════════════════════════════════════════════════
        # UPPER BOUND: Miglior soluzione trovata finora
        # Inizializzato con il costo del Greedy per potare più velocemente
        # ══════════════════════════════════════════════════════════════
        self.best_cost = initial_upper_bound
        self.best_route = None
        
        # Statistiche per analisi
        self.nodes_explored = 0
        self.nodes_pruned = 0
    
    def solve(self, start: int = 0) -> Tuple[List[int], float]:
        """
        Avvia la ricerca Branch and Bound.
        
        Args:
            start: Nodo di partenza (deposito)
        
        Returns:
            Tupla (percorso_ottimo, costo_ottimo)
        """
        # Stato iniziale: solo il deposito visitato
        visited = [False] * self.n
        visited[start] = True
        current_route = [start]
        
        print(f"\n🔍 Avvio Branch and Bound con Upper Bound iniziale: {self.best_cost:.2f} km")
        
        # Chiamata ricorsiva
        self._branch_and_bound(
            current_node=start,
            current_route=current_route,
            current_cost=0.0,
            visited=visited,
            depth=1
        )
        
        # Aggiungi il ritorno al deposito nel percorso finale
        if self.best_route and self.best_route[-1] != start:
            self.best_route.append(start)
        
        return self.best_route, self.best_cost
    
    def _branch_and_bound(
        self,
        current_node: int,
        current_route: List[int],
        current_cost: float,
        visited: List[bool],
        depth: int
    ):
        """
        Funzione ricorsiva che implementa Branch and Bound.
        
        ALGORITMO (Pseudocodice):
        ─────────────────────────
        function B&B(nodo_corrente, percorso, costo, visitati, profondità):
            
            # CASO BASE: Tutti i nodi visitati
            if profondità == N:
                costo_totale = costo + dist[nodo_corrente][deposito]
                if costo_totale < best_cost:
                    best_cost = costo_totale
                    best_route = percorso
                return
            
            # ══════════════════════════════════════════════════════════
            # BRANCHING (Ramificazione - Cap. 2.3.1)
            # Genera tutti i possibili nodi figli (nodi non ancora visitati)
            # ══════════════════════════════════════════════════════════
            for ogni nodo j non visitato:
                nuovo_costo = costo + dist[nodo_corrente][j]
                
                # ══════════════════════════════════════════════════════
                # PRUNING (Potatura - Cap. 2.2.6)
                # Se il costo parziale >= Upper Bound, taglia il ramo
                # ══════════════════════════════════════════════════════
                if nuovo_costo >= best_cost:
                    continue  # PRUNE!
                
                # Esplora il ramo
                B&B(j, percorso + [j], nuovo_costo, visitati ∪ {j}, profondità + 1)
        
        Args:
            current_node: Nodo attuale
            current_route: Percorso costruito finora
            current_cost: Costo accumulato finora
            visited: Array booleano dei nodi visitati
            depth: Profondità nell'albero (= numero di nodi nel percorso)
        """
        self.nodes_explored += 1
        
        # ══════════════════════════════════════════════════════════════
        # CASO BASE: Tutti i nodi sono stati visitati
        # Calcola il costo totale includendo il ritorno al deposito
        # ══════════════════════════════════════════════════════════════
        if depth == self.n:
            # Costo per tornare al deposito (nodo 0)
            total_cost = current_cost + self.dist_matrix[current_node][0]
            
            # Aggiorna la soluzione migliore se trovata
            if total_cost < self.best_cost:
                self.best_cost = total_cost
                self.best_route = current_route.copy()
                print(f"   💡 Nuova soluzione migliore trovata: {total_cost:.2f} km")
            return
        
        # ══════════════════════════════════════════════════════════════
        # BRANCHING (Ramificazione) - Capitolo 2.3.1
        # ══════════════════════════════════════════════════════════════
        # Genera tutti i possibili "figli" del nodo corrente
        # Ogni figlio corrisponde a un nodo non ancora visitato
        # L'ordine di esplorazione può influenzare l'efficienza
        # ══════════════════════════════════════════════════════════════
        
        # Ottimizzazione: ordina i figli per distanza crescente (esplora prima i più vicini)
        candidates = []
        for next_node in range(self.n):
            if not visited[next_node]:
                candidates.append((next_node, self.dist_matrix[current_node][next_node]))
        
        # Ordina per distanza (euristica per trovare buone soluzioni prima)
        candidates.sort(key=lambda x: x[1])
        
        for next_node, edge_cost in candidates:
            new_cost = current_cost + edge_cost
            
            # ══════════════════════════════════════════════════════════
            # PRUNING (Potatura) - Capitolo 2.2.6
            # ══════════════════════════════════════════════════════════
            # REGOLA DI PRUNING:
            # Se il costo parziale è già >= al miglior costo totale trovato,
            # NON ha senso continuare ad esplorare questo ramo perché
            # qualsiasi soluzione completa in questo sottoalbero avrà
            # costo >= new_cost > best_cost
            # ══════════════════════════════════════════════════════════
            if new_cost >= self.best_cost:
                self.nodes_pruned += 1
                continue  # ✂️ PRUNE! Taglia questo ramo
            
            # Il ramo è promettente: esplora ricorsivamente
            visited[next_node] = True
            current_route.append(next_node)
            
            self._branch_and_bound(
                current_node=next_node,
                current_route=current_route,
                current_cost=new_cost,
                visited=visited,
                depth=depth + 1
            )
            
            # Backtracking: ripristina lo stato per esplorare altri rami
            visited[next_node] = False
            current_route.pop()


# ==============================================================================
# SEZIONE 4: CARICAMENTO DATI E BENCHMARK
# ==============================================================================

def load_route_data(filepath: str, max_nodes: int = 14) -> Tuple[List[Node], Node]:
    """
    Carica i dati dal CSV e crea la lista di nodi.
    
    Args:
        filepath: Percorso del file CSV
        max_nodes: Numero massimo di nodi da caricare (escluso deposito)
                   IMPORTANTE: Limitare a 12-14 per evitare tempi esponenziali
    
    Returns:
        Tupla (lista_nodi, deposito)
    """
    df = pd.read_csv(filepath)
    
    nodes = []
    depot = None
    
    for idx, row in df.iterrows():
        node_type = row.get('Node_Type', '')
        
        if node_type == 'Depot' and depot is None:
            depot = Node(
                id=0,
                latitude=row['Latitude'],
                longitude=row['Longitude'],
                name='DEPOT'
            )
        elif node_type == 'Delivery' and len(nodes) < max_nodes:
            nodes.append(Node(
                id=len(nodes) + 1,  # ID 1, 2, 3, ...
                latitude=row['Latitude'],
                longitude=row['Longitude'],
                name=str(row.get('Package_ID', f'PKG_{idx}'))
            ))
    
    # Inserisci il deposito all'inizio (indice 0)
    if depot:
        all_nodes = [depot] + nodes
    else:
        # Fallback: usa il primo nodo come deposito
        print("⚠️ Deposito non trovato, uso il primo nodo")
        all_nodes = nodes
        all_nodes[0].name = 'DEPOT (fallback)'
    
    return all_nodes, depot


def run_benchmark(filepath: str = "route_van_0.csv", max_nodes: int = 12):
    """
    Esegue il benchmark completo: Greedy vs Branch and Bound.
    
    Args:
        filepath: Percorso del CSV con il percorso
        max_nodes: Numero massimo di nodi per B&B (default: 12)
    """
    print("\n" + "="*70)
    print("  BENCHMARK: Greedy Best-First Search vs Branch and Bound")
    print("  Problema: Traveling Salesman Problem (TSP)")
    print("="*70)
    
    # ─────────────────────────────────────────────────────────────
    # STEP 1: Caricamento dati
    # ─────────────────────────────────────────────────────────────
    print(f"\n📂 Caricamento dati da: {filepath}")
    print(f"   Limite nodi per B&B: {max_nodes} (+ deposito)")
    
    try:
        nodes, depot = load_route_data(filepath, max_nodes=max_nodes)
    except FileNotFoundError:
        print(f"❌ File non trovato: {filepath}")
        print("   Assicurati di aver eseguito search.ipynb prima.")
        return
    
    n = len(nodes)
    print(f"   Nodi caricati: {n} (1 deposito + {n-1} consegne)")
    
    if n < 3:
        print("❌ Errore: Servono almeno 3 nodi per il TSP")
        return
    
    # ─────────────────────────────────────────────────────────────
    # STEP 2: Pre-calcolo matrice delle distanze
    # ─────────────────────────────────────────────────────────────
    print(f"\n📐 Pre-calcolo matrice distanze {n}x{n}...")
    dist_matrix = build_distance_matrix(nodes)
    print(f"   Matrice calcolata: {dist_matrix.shape}")
    
    # ─────────────────────────────────────────────────────────────
    # STEP 3: Esecuzione GREEDY (Baseline)
    # ─────────────────────────────────────────────────────────────
    print("\n" + "-"*70)
    print("🏃 ALGORITMO 1: Greedy Best-First Search (Baseline)")
    print("-"*70)
    
    start_time = time.time()
    greedy_route, greedy_cost = greedy_tsp(dist_matrix)
    greedy_time = time.time() - start_time
    
    print(f"   Percorso: {greedy_route}")
    print(f"   Distanza: {greedy_cost:.2f} km")
    print(f"   Tempo: {greedy_time:.6f} sec")
    
    # ─────────────────────────────────────────────────────────────
    # STEP 4: Esecuzione BRANCH AND BOUND (Challenger)
    # ─────────────────────────────────────────────────────────────
    print("\n" + "-"*70)
    print("🔬 ALGORITMO 2: Branch and Bound (Challenger)")
    print("-"*70)
    
    # OTTIMIZZAZIONE: Usa il costo del Greedy come Upper Bound iniziale
    # Questo permette di potare molti più rami fin dall'inizio!
    bnb_solver = BranchAndBoundTSP(dist_matrix, initial_upper_bound=greedy_cost)
    
    start_time = time.time()
    bnb_route, bnb_cost = bnb_solver.solve()
    bnb_time = time.time() - start_time
    
    print(f"\n   Percorso ottimo: {bnb_route}")
    print(f"   Distanza ottima: {bnb_cost:.2f} km")
    print(f"   Tempo: {bnb_time:.4f} sec")
    print(f"   Nodi esplorati: {bnb_solver.nodes_explored:,}")
    print(f"   Nodi potati (pruned): {bnb_solver.nodes_pruned:,}")
    
    # ─────────────────────────────────────────────────────────────
    # STEP 5: Tabella comparativa
    # ─────────────────────────────────────────────────────────────
    print("\n" + "="*70)
    print("  TABELLA COMPARATIVA")
    print("="*70)
    
    # Calcolo risparmio
    savings_km = greedy_cost - bnb_cost
    savings_pct = (savings_km / greedy_cost * 100) if greedy_cost > 0 else 0
    speedup = greedy_time / bnb_time if bnb_time > 0 else float('inf')
    
    # Tabella dei risultati
    table_data = [
        ["Greedy (GBFS)", f"{greedy_cost:.2f}", f"{greedy_time:.6f}", "Baseline"],
        ["Branch & Bound", f"{bnb_cost:.2f}", f"{bnb_time:.4f}", "Ottimo ✓"],
    ]
    
    headers = ["Algoritmo", "Distanza (km)", "Tempo (sec)", "Tipo"]
    print("\n" + tabulate(table_data, headers=headers, tablefmt="grid"))
    
    # Statistiche aggiuntive
    print("\n" + "-"*70)
    print("  ANALISI COMPARATIVA")
    print("-"*70)
    stats_data = [
        ["Risparmio B&B vs Greedy", f"{savings_km:.2f} km", f"{savings_pct:.2f}%"],
        ["Rapporto tempi (Greedy/B&B)", f"{speedup:.2f}x", 
         "Greedy più veloce" if speedup > 1 else "B&B più veloce"],
        ["Nodi esplorati (B&B)", f"{bnb_solver.nodes_explored:,}", ""],
        ["Nodi potati (B&B)", f"{bnb_solver.nodes_pruned:,}", 
         f"{bnb_solver.nodes_pruned/(bnb_solver.nodes_explored+bnb_solver.nodes_pruned)*100:.1f}% efficienza pruning"],
        ["Complessità TSP (N!)", f"{factorial(n-1):,}", "permutazioni teoriche"],
    ]
    
    stats_headers = ["Metrica", "Valore", "Note"]
    print("\n" + tabulate(stats_data, headers=stats_headers, tablefmt="grid"))
    
    # ─────────────────────────────────────────────────────────────
    # STEP 6: Interpretazione dei risultati
    # ─────────────────────────────────────────────────────────────
    print("\n" + "="*70)
    print("  INTERPRETAZIONE RISULTATI")
    print("="*70)
    
    if savings_pct > 0:
        print(f"""
    ✅ Branch and Bound ha trovato un percorso MIGLIORE del Greedy!
    
    📊 Il Greedy ha commesso errori di {savings_pct:.2f}% a causa della sua
       natura "miope": sceglie sempre l'opzione localmente ottimale
       senza considerare le conseguenze globali.
    
    🔬 Branch and Bound garantisce l'OTTIMO GLOBALE esplorando
       sistematicamente lo spazio delle soluzioni e potando i rami
       non promettenti grazie alla tecnica di pruning.
    """)
    else:
        print(f"""
    🎯 In questo caso, il Greedy ha trovato casualmente la soluzione ottimale!
    
    ⚠️  Attenzione: questo NON è garantito in generale.
        Su istanze più complesse, il Greedy può essere significativamente
        peggiore dell'ottimo.
    """)
    
    print(f"""
    ⏱️  Trade-off Tempo/Qualità:
        - Greedy: O(N²) - molto veloce, ma soluzione approssimata
        - B&B: O(N!) nel caso peggiore - lento, ma soluzione ottima
        
        Con {n} nodi, il numero teorico di permutazioni è {factorial(n-1):,}
        Ma grazie al pruning, B&B ha esplorato solo {bnb_solver.nodes_explored:,} nodi
        ({bnb_solver.nodes_explored/factorial(n-1)*100:.4f}% dello spazio totale)
    """)
    
    print("="*70)
    print("  BENCHMARK COMPLETATO")
    print("="*70)
    
    return {
        'greedy': {'route': greedy_route, 'cost': greedy_cost, 'time': greedy_time},
        'bnb': {'route': bnb_route, 'cost': bnb_cost, 'time': bnb_time,
                'nodes_explored': bnb_solver.nodes_explored,
                'nodes_pruned': bnb_solver.nodes_pruned},
        'savings_pct': savings_pct
    }


# ==============================================================================
# MAIN
# ==============================================================================

if __name__ == "__main__":
    # Configurazione benchmark
    # NOTA: Aumentare max_nodes oltre 14-15 può richiedere tempi molto lunghi!
    
    results = run_benchmark(
        filepath="route_van_0.csv",
        max_nodes=12  # Limitato per evitare tempi esponenziali
    )


  BENCHMARK: Greedy Best-First Search vs Branch and Bound
  Problema: Traveling Salesman Problem (TSP)

📂 Caricamento dati da: route_van_0.csv
   Limite nodi per B&B: 12 (+ deposito)
   Nodi caricati: 13 (1 deposito + 12 consegne)

📐 Pre-calcolo matrice distanze 13x13...
   Matrice calcolata: (13, 13)

----------------------------------------------------------------------
🏃 ALGORITMO 1: Greedy Best-First Search (Baseline)
----------------------------------------------------------------------
   Percorso: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 0]
   Distanza: 51.93 km
   Tempo: 0.000019 sec

----------------------------------------------------------------------
🔬 ALGORITMO 2: Branch and Bound (Challenger)
----------------------------------------------------------------------

🔍 Avvio Branch and Bound con Upper Bound iniziale: 51.93 km
   💡 Nuova soluzione migliore trovata: 51.20 km
   💡 Nuova soluzione migliore trovata: 51.13 km
   💡 Nuova soluzione migliore trovata: 51.07 km

   P

# A*

In [ ]:
"""
==============================================================================
MODULO DI RICERCA: A* Algorithm per TSP
==============================================================================
Corso: Intelligenza Artificiale - Ricerca Operativa
Riferimenti Teorici:
  - Capitolo 2.2.4: Algoritmo A*
  - Capitolo 2.2.2: Funzioni Euristiche Ammissibili

DESCRIZIONE:
Implementazione dell'algoritmo A* per il Problema del Commesso Viaggiatore (TSP).
A* combina il costo effettivo g(n) con una stima euristica h(n) per trovare
il percorso ottimale in modo più efficiente rispetto alla ricerca esaustiva.

FORMULA A*:
  f(n) = g(n) + h(n)
  dove:
    - g(n): costo effettivo dal nodo iniziale al nodo n
    - h(n): stima euristica del costo da n al goal
    - f(n): costo totale stimato del percorso attraverso n

GARANZIA DI OTTIMALITÀ:
  A* è ottimale SE l'euristica h(n) è AMMISSIBILE (non sovrastima mai il costo reale)
==============================================================================
"""

import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2, factorial
from typing import List, Tuple, Optional, Set, FrozenSet
from dataclasses import dataclass, field
import heapq
import time
from tabulate import tabulate


# ==============================================================================
# SEZIONE 1: STRUTTURE DATI
# ==============================================================================

@dataclass
class Node:
    """Rappresenta un nodo (punto di consegna o deposito)."""
    id: int
    latitude: float
    longitude: float
    name: str = ""
    
    def __repr__(self):
        return f"Node({self.id}: {self.name})"


@dataclass(order=True)
class AStarState:
    """
    Stato per la coda di priorità di A*.
    
    Ogni stato rappresenta un percorso parziale nel TSP.
    L'ordinamento è basato su f_cost per la coda di priorità.
    """
    f_cost: float  # f(n) = g(n) + h(n) - usato per ordinamento
    g_cost: float = field(compare=False)  # Costo effettivo finora
    current_node: int = field(compare=False)  # Nodo corrente
    path: Tuple[int, ...] = field(compare=False)  # Percorso finora
    visited: FrozenSet[int] = field(compare=False)  # Nodi visitati


# ==============================================================================
# SEZIONE 2: FUNZIONI DI UTILITÀ
# ==============================================================================

def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Calcola la distanza in km tra due punti usando la formula di Haversine.
    Tiene conto della curvatura terrestre.
    """
    R = 6371.0  # Raggio della Terra in km
    
    lat1_rad, lat2_rad = radians(lat1), radians(lat2)
    delta_lat = radians(lat2 - lat1)
    delta_lon = radians(lon2 - lon1)
    
    a = sin(delta_lat / 2)**2 + cos(lat1_rad) * cos(lat2_rad) * sin(delta_lon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    
    return R * c


def build_distance_matrix(nodes: List[Node]) -> np.ndarray:
    """
    Pre-calcola la matrice NxN delle distanze Haversine.
    
    Args:
        nodes: Lista di nodi (indice 0 = deposito)
    
    Returns:
        Matrice numpy NxN con le distanze in km
    """
    n = len(nodes)
    dist_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i + 1, n):
            d = haversine_distance(
                nodes[i].latitude, nodes[i].longitude,
                nodes[j].latitude, nodes[j].longitude
            )
            dist_matrix[i][j] = d
            dist_matrix[j][i] = d  # Simmetria
    
    return dist_matrix


# ==============================================================================
# SEZIONE 3: FUNZIONI EURISTICHE PER A*
# ==============================================================================

def mst_heuristic(dist_matrix: np.ndarray, unvisited: Set[int], current: int, start: int) -> float:
    """
    Euristica MST (Minimum Spanning Tree) per TSP.
    
    TEORIA (Capitolo 2.2.2):
    L'euristica MST calcola il costo del Minimum Spanning Tree che collega
    tutti i nodi non ancora visitati più il nodo corrente e il deposito.
    
    AMMISSIBILITÀ:
    L'MST è ammissibile perché il costo dell'MST è sempre ≤ al costo del
    tour ottimale sui nodi rimanenti (l'MST è un lower bound).
    
    Args:
        dist_matrix: Matrice delle distanze
        unvisited: Set dei nodi non ancora visitati
        current: Nodo corrente
        start: Nodo di partenza (deposito)
    
    Returns:
        Stima del costo minimo per completare il tour
    """
    if not unvisited:
        # Se non ci sono nodi da visitare, ritorna la distanza al deposito
        return dist_matrix[current][start]
    
    # Nodi da considerare per l'MST: non visitati + deposito (per il ritorno)
    nodes_for_mst = list(unvisited) + [start]
    
    if len(nodes_for_mst) == 1:
        # Solo il deposito rimasto
        return dist_matrix[current][start]
    
    # Algoritmo di Prim per MST
    n_mst = len(nodes_for_mst)
    in_mst = [False] * n_mst
    min_cost = [float('inf')] * n_mst
    
    # Inizia dal primo nodo
    min_cost[0] = 0
    mst_cost = 0
    
    for _ in range(n_mst):
        # Trova il nodo con costo minimo non ancora nell'MST
        min_val = float('inf')
        min_idx = -1
        for i in range(n_mst):
            if not in_mst[i] and min_cost[i] < min_val:
                min_val = min_cost[i]
                min_idx = i
        
        if min_idx == -1:
            break
        
        in_mst[min_idx] = True
        mst_cost += min_val
        
        # Aggiorna i costi per i nodi adiacenti
        for i in range(n_mst):
            if not in_mst[i]:
                edge_cost = dist_matrix[nodes_for_mst[min_idx]][nodes_for_mst[i]]
                if edge_cost < min_cost[i]:
                    min_cost[i] = edge_cost
    
    # Aggiungi la distanza minima dal nodo corrente ai nodi non visitati
    min_to_unvisited = min(dist_matrix[current][u] for u in unvisited)
    
    return mst_cost + min_to_unvisited


def nearest_neighbor_heuristic(dist_matrix: np.ndarray, unvisited: Set[int], 
                                current: int, start: int) -> float:
    """
    Euristica del Nearest Neighbor come lower bound.
    
    Calcola una stima sommando:
    1. Distanza minima dal nodo corrente a un nodo non visitato
    2. Distanza minima da ogni nodo non visitato al suo vicino più prossimo
    3. Distanza minima dall'ultimo nodo al deposito
    
    NOTA: Questa euristica è più veloce di MST ma meno accurata.
    
    Args:
        dist_matrix: Matrice delle distanze
        unvisited: Set dei nodi non ancora visitati
        current: Nodo corrente
        start: Nodo di partenza (deposito)
    
    Returns:
        Stima del costo minimo per completare il tour
    """
    if not unvisited:
        return dist_matrix[current][start]
    
    h = 0.0
    
    # Distanza minima dal nodo corrente ai non visitati
    min_from_current = min(dist_matrix[current][u] for u in unvisited)
    h += min_from_current
    
    # Per ogni nodo non visitato, aggiungi la distanza minima al vicino
    for u in unvisited:
        # Considera sia gli altri non visitati che il deposito
        candidates = [v for v in unvisited if v != u] + [start]
        if candidates:
            min_from_u = min(dist_matrix[u][c] for c in candidates)
            h += min_from_u
    
    # Dividi per 2 perché abbiamo contato ogni arco due volte
    return h / 2


def simple_heuristic(dist_matrix: np.ndarray, unvisited: Set[int], 
                     current: int, start: int) -> float:
    """
    Euristica semplice: distanza minima per uscire + distanza minima per tornare.
    
    Molto veloce ma meno informativa.
    
    Args:
        dist_matrix: Matrice delle distanze
        unvisited: Set dei nodi non ancora visitati
        current: Nodo corrente
        start: Nodo di partenza (deposito)
    
    Returns:
        Stima del costo minimo per completare il tour
    """
    if not unvisited:
        return dist_matrix[current][start]
    
    # Distanza minima dal corrente a un non visitato
    min_out = min(dist_matrix[current][u] for u in unvisited)
    
    # Distanza minima da un non visitato al deposito
    min_back = min(dist_matrix[u][start] for u in unvisited)
    
    return min_out + min_back


# ==============================================================================
# SEZIONE 4: ALGORITMO A* PER TSP
# ==============================================================================

class AStarTSP:
    """
    Implementazione dell'algoritmo A* per il Problema del Commesso Viaggiatore.
    
    ALGORITMO A* (Capitolo 2.2.4):
    1. Inizializza la coda di priorità con lo stato iniziale
    2. Estrai lo stato con f(n) minimo
    3. Se è uno stato goal, ritorna la soluzione
    4. Altrimenti, espandi generando tutti gli stati successori
    5. Per ogni successore, calcola f(n) = g(n) + h(n) e aggiungi alla coda
    6. Ripeti dal passo 2
    
    PROPRIETÀ:
    - Completo: Trova sempre una soluzione se esiste
    - Ottimale: Se h(n) è ammissibile, trova la soluzione ottimale
    - Efficienza: Dipende dalla qualità dell'euristica
    """
    
    def __init__(self, dist_matrix: np.ndarray, heuristic: str = "mst"):
        """
        Inizializza il solver A*.
        
        Args:
            dist_matrix: Matrice delle distanze pre-calcolata
            heuristic: Tipo di euristica ("mst", "nn", "simple")
        """
        self.dist_matrix = dist_matrix
        self.n = len(dist_matrix)
        self.heuristic_type = heuristic
        
        # Seleziona la funzione euristica
        if heuristic == "mst":
            self.heuristic = mst_heuristic
        elif heuristic == "nn":
            self.heuristic = nearest_neighbor_heuristic
        else:
            self.heuristic = simple_heuristic
        
        # Statistiche
        self.nodes_expanded = 0
        self.nodes_generated = 0
        self.max_frontier_size = 0
    
    def solve(self, start: int = 0) -> Tuple[List[int], float]:
        """
        Esegue l'algoritmo A* per trovare il tour ottimale.
        
        PSEUDOCODICE A* per TSP:
        ─────────────────────────
        function A*_TSP(start):
            open_list ← priority_queue()
            
            # Stato iniziale
            initial_state ← (start, {start}, [start], 0)
            h ← heuristic(initial_state)
            open_list.push((h, initial_state))
            
            while open_list non vuota:
                current ← open_list.pop()  # Stato con f minimo
                
                # Goal test: tutti i nodi visitati
                if |current.visited| == N:
                    return current.path + [start], current.g + dist[current.node][start]
                
                # Espansione: genera successori
                for each neighbor not in current.visited:
                    new_g ← current.g + dist[current.node][neighbor]
                    new_h ← heuristic(neighbor, remaining_nodes)
                    new_f ← new_g + new_h
                    new_state ← (neighbor, current.visited ∪ {neighbor}, 
                                 current.path + [neighbor], new_g)
                    open_list.push((new_f, new_state))
            
            return None  # Nessuna soluzione trovata
        
        Args:
            start: Nodo di partenza (deposito), default 0
        
        Returns:
            Tupla (percorso_ottimo, costo_ottimo)
        """
        print(f"\n🔍 Avvio A* con euristica: {self.heuristic_type.upper()}")
        
        # ══════════════════════════════════════════════════════════════
        # INIZIALIZZAZIONE
        # ══════════════════════════════════════════════════════════════
        
        # Stato iniziale: solo il deposito visitato
        initial_visited = frozenset([start])
        all_nodes = set(range(self.n))
        unvisited = all_nodes - initial_visited
        
        # Calcola euristica iniziale
        h_initial = self.heuristic(self.dist_matrix, unvisited, start, start)
        
        # Crea lo stato iniziale
        initial_state = AStarState(
            f_cost=h_initial,
            g_cost=0.0,
            current_node=start,
            path=(start,),
            visited=initial_visited
        )
        
        # Coda di priorità (min-heap basato su f_cost)
        open_list = [initial_state]
        heapq.heapify(open_list)
        
        # Dizionario per tenere traccia del miglior g_cost per ogni stato
        # Chiave: (nodo_corrente, visited_set)
        best_g = {(start, initial_visited): 0.0}
        
        self.nodes_expanded = 0
        self.nodes_generated = 1
        self.max_frontier_size = 1
        
        # ══════════════════════════════════════════════════════════════
        # LOOP PRINCIPALE A*
        # ══════════════════════════════════════════════════════════════
        
        while open_list:
            # Aggiorna statistiche sulla frontiera
            self.max_frontier_size = max(self.max_frontier_size, len(open_list))
            
            # Estrai lo stato con f(n) minimo
            current = heapq.heappop(open_list)
            self.nodes_expanded += 1
            
            # ══════════════════════════════════════════════════════════
            # GOAL TEST: Tutti i nodi sono stati visitati
            # ══════════════════════════════════════════════════════════
            if len(current.visited) == self.n:
                # Calcola il costo totale includendo il ritorno al deposito
                return_cost = self.dist_matrix[current.current_node][start]
                total_cost = current.g_cost + return_cost
                final_path = list(current.path) + [start]
                
                print(f"   ✅ Soluzione trovata dopo {self.nodes_expanded} espansioni")
                
                return final_path, total_cost
            
            # Verifica se questo stato è già stato raggiunto con costo migliore
            state_key = (current.current_node, current.visited)
            if state_key in best_g and best_g[state_key] < current.g_cost:
                continue  # Salta questo stato, ne abbiamo già uno migliore
            
            # ══════════════════════════════════════════════════════════
            # ESPANSIONE: Genera tutti i successori
            # ══════════════════════════════════════════════════════════
            unvisited_nodes = all_nodes - current.visited
            
            for next_node in unvisited_nodes:
                # Calcola g(n) per il nuovo stato
                edge_cost = self.dist_matrix[current.current_node][next_node]
                new_g = current.g_cost + edge_cost
                
                # Nuovo insieme di nodi visitati
                new_visited = current.visited | {next_node}
                
                # Verifica se abbiamo già raggiunto questo stato con costo migliore
                new_state_key = (next_node, new_visited)
                if new_state_key in best_g and best_g[new_state_key] <= new_g:
                    continue  # Salta, abbiamo già uno stato migliore
                
                # Aggiorna il miglior costo per questo stato
                best_g[new_state_key] = new_g
                
                # Calcola h(n) per il nuovo stato
                remaining = all_nodes - new_visited
                new_h = self.heuristic(self.dist_matrix, remaining, next_node, start)
                
                # Calcola f(n) = g(n) + h(n)
                new_f = new_g + new_h
                
                # Crea il nuovo stato
                new_state = AStarState(
                    f_cost=new_f,
                    g_cost=new_g,
                    current_node=next_node,
                    path=current.path + (next_node,),
                    visited=new_visited
                )
                
                # Aggiungi alla coda di priorità
                heapq.heappush(open_list, new_state)
                self.nodes_generated += 1
            
            # Log periodico del progresso
            if self.nodes_expanded % 1000 == 0:
                print(f"   Espansi: {self.nodes_expanded}, Frontiera: {len(open_list)}")
        
        # Nessuna soluzione trovata (non dovrebbe mai accadere per TSP connesso)
        print("   ❌ Nessuna soluzione trovata!")
        return None, float('inf')


# ==============================================================================
# SEZIONE 5: ALGORITMO GREEDY (BASELINE)
# ==============================================================================

def greedy_tsp(dist_matrix: np.ndarray, start: int = 0) -> Tuple[List[int], float]:
    """
    Algoritmo Greedy per TSP: ad ogni passo sceglie il nodo più vicino.
    
    Args:
        dist_matrix: Matrice delle distanze pre-calcolata
        start: Indice del nodo di partenza
    
    Returns:
        Tupla (percorso, costo_totale)
    """
    n = len(dist_matrix)
    visited = [False] * n
    route = [start]
    visited[start] = True
    total_cost = 0.0
    current = start
    
    for _ in range(n - 1):
        best_next = -1
        best_dist = float('inf')
        
        for j in range(n):
            if not visited[j] and dist_matrix[current][j] < best_dist:
                best_dist = dist_matrix[current][j]
                best_next = j
        
        if best_next != -1:
            route.append(best_next)
            visited[best_next] = True
            total_cost += best_dist
            current = best_next
    
    total_cost += dist_matrix[current][start]
    route.append(start)
    
    return route, total_cost


# ==============================================================================
# SEZIONE 6: CARICAMENTO DATI
# ==============================================================================

def load_route_data(filepath: str, max_nodes: int = 10) -> Tuple[List[Node], Node]:
    """
    Carica i dati dal CSV e crea la lista di nodi.
    
    Args:
        filepath: Percorso del file CSV
        max_nodes: Numero massimo di nodi da caricare (escluso deposito)
    
    Returns:
        Tupla (lista_nodi, deposito)
    """
    df = pd.read_csv(filepath)
    
    nodes = []
    depot = None
    
    for idx, row in df.iterrows():
        node_type = row.get('Node_Type', '')
        
        if node_type == 'Depot' and depot is None:
            depot = Node(
                id=0,
                latitude=row['Latitude'],
                longitude=row['Longitude'],
                name='DEPOT'
            )
        elif node_type == 'Delivery' and len(nodes) < max_nodes:
            nodes.append(Node(
                id=len(nodes) + 1,
                latitude=row['Latitude'],
                longitude=row['Longitude'],
                name=str(row.get('Package_ID', f'PKG_{idx}'))
            ))
    
    if depot:
        all_nodes = [depot] + nodes
    else:
        print("⚠️ Deposito non trovato, uso il primo nodo")
        all_nodes = nodes
        all_nodes[0].name = 'DEPOT (fallback)'
    
    return all_nodes, depot


# ==============================================================================
# SEZIONE 7: BENCHMARK A* vs GREEDY vs BRANCH AND BOUND
# ==============================================================================

def run_astar_benchmark(filepath: str = "route_van_0.csv", max_nodes: int = 10):
    """
    Esegue il benchmark completo: A* vs Greedy.
    
    Args:
        filepath: Percorso del CSV con il percorso
        max_nodes: Numero massimo di nodi per A* (default: 10)
    """
    print("\n" + "="*70)
    print("  BENCHMARK: A* Algorithm per Traveling Salesman Problem")
    print("  Corso: Intelligenza Artificiale - Ricerca Informata")
    print("="*70)
    
    # ─────────────────────────────────────────────────────────────
    # STEP 1: Caricamento dati
    # ─────────────────────────────────────────────────────────────
    print(f"\n📂 Caricamento dati da: {filepath}")
    print(f"   Limite nodi: {max_nodes} (+ deposito)")
    
    try:
        nodes, depot = load_route_data(filepath, max_nodes=max_nodes)
    except FileNotFoundError:
        print(f"❌ File non trovato: {filepath}")
        print("   Assicurati di aver eseguito search.ipynb prima.")
        return
    
    n = len(nodes)
    print(f"   Nodi caricati: {n} (1 deposito + {n-1} consegne)")
    
    if n < 3:
        print("❌ Errore: Servono almeno 3 nodi per il TSP")
        return
    
    # ─────────────────────────────────────────────────────────────
    # STEP 2: Pre-calcolo matrice delle distanze
    # ─────────────────────────────────────────────────────────────
    print(f"\n📐 Pre-calcolo matrice distanze {n}x{n}...")
    dist_matrix = build_distance_matrix(nodes)
    
    # ─────────────────────────────────────────────────────────────
    # STEP 3: Esecuzione GREEDY (Baseline)
    # ─────────────────────────────────────────────────────────────
    print("\n" + "-"*70)
    print("🏃 ALGORITMO 1: Greedy Best-First Search (Baseline)")
    print("-"*70)
    
    start_time = time.time()
    greedy_route, greedy_cost = greedy_tsp(dist_matrix)
    greedy_time = time.time() - start_time
    
    print(f"   Percorso: {greedy_route}")
    print(f"   Distanza: {greedy_cost:.2f} km")
    print(f"   Tempo: {greedy_time:.6f} sec")
    
    # ─────────────────────────────────────────────────────────────
    # STEP 4: Esecuzione A* con diverse euristiche
    # ─────────────────────────────────────────────────────────────
    results = {
        'greedy': {
            'route': greedy_route,
            'cost': greedy_cost,
            'time': greedy_time,
            'expanded': 'N/A',
            'generated': 'N/A'
        }
    }
    
    heuristics = [("mst", "MST (Minimum Spanning Tree)"), 
                  ("nn", "Nearest Neighbor"), 
                  ("simple", "Simple (Min In + Min Out)")]
    
    for h_type, h_name in heuristics:
        print("\n" + "-"*70)
        print(f"🔬 ALGORITMO A* con Euristica: {h_name}")
        print("-"*70)
        
        astar_solver = AStarTSP(dist_matrix, heuristic=h_type)
        
        start_time = time.time()
        astar_route, astar_cost = astar_solver.solve()
        astar_time = time.time() - start_time
        
        print(f"\n   Percorso: {astar_route}")
        print(f"   Distanza: {astar_cost:.2f} km")
        print(f"   Tempo: {astar_time:.4f} sec")
        print(f"   Nodi espansi: {astar_solver.nodes_expanded:,}")
        print(f"   Nodi generati: {astar_solver.nodes_generated:,}")
        print(f"   Max frontiera: {astar_solver.max_frontier_size:,}")
        
        results[f'astar_{h_type}'] = {
            'route': astar_route,
            'cost': astar_cost,
            'time': astar_time,
            'expanded': astar_solver.nodes_expanded,
            'generated': astar_solver.nodes_generated,
            'max_frontier': astar_solver.max_frontier_size
        }
    
    # ─────────────────────────────────────────────────────────────
    # STEP 5: Tabella comparativa
    # ─────────────────────────────────────────────────────────────
    print("\n" + "="*70)
    print("  TABELLA COMPARATIVA")
    print("="*70)
    
    table_data = [
        ["Greedy (GBFS)", 
         f"{results['greedy']['cost']:.2f}", 
         f"{results['greedy']['time']:.6f}", 
         "N/A", 
         "Sub-ottimale"],
    ]
    
    for h_type, h_name in heuristics:
        key = f'astar_{h_type}'
        optimal_flag = "✓ Ottimale" if results[key]['cost'] <= min(
            results[f'astar_{h[0]}']['cost'] for h in heuristics
        ) else ""
        table_data.append([
            f"A* ({h_type.upper()})",
            f"{results[key]['cost']:.2f}",
            f"{results[key]['time']:.4f}",
            f"{results[key]['expanded']:,}",
            optimal_flag
        ])
    
    headers = ["Algoritmo", "Distanza (km)", "Tempo (sec)", "Nodi Espansi", "Note"]
    print("\n" + tabulate(table_data, headers=headers, tablefmt="grid"))
    
    # ─────────────────────────────────────────────────────────────
    # STEP 6: Analisi comparativa
    # ─────────────────────────────────────────────────────────────
    best_astar = min(results[f'astar_{h[0]}']['cost'] for h in heuristics)
    savings = greedy_cost - best_astar
    savings_pct = (savings / greedy_cost * 100) if greedy_cost > 0 else 0
    
    print("\n" + "-"*70)
    print("  ANALISI COMPARATIVA")
    print("-"*70)
    
    print(f"""
    📊 CONFRONTO A* vs GREEDY:
    
    • Distanza Greedy:  {greedy_cost:.2f} km
    • Distanza A*:      {best_astar:.2f} km
    • Risparmio:        {savings:.2f} km ({savings_pct:.2f}%)
    
    
    A* utilizza f(n) = g(n) + h(n) dove:
    - g(n): costo reale dal nodo iniziale
    - h(n): stima euristica al goal
    
    ✓ OTTIMALITÀ: A* è ottimale se h(n) è ammissibile
      (non sovrastima mai il costo reale)
    
    ✓ COMPLETEZZA: A* trova sempre una soluzione se esiste
    
    📈 EFFICIENZA DELLE EURISTICHE:
    - MST: Più accurata ma più costosa da calcolare
    - NN:  Buon compromesso tra accuratezza e velocità  
    - Simple: Veloce ma meno informativa
    
    💡 OSSERVAZIONE:
    Con {n} nodi, lo spazio delle soluzioni ha {factorial(n-1):,} permutazioni.
    A* esplora solo una frazione di questo spazio grazie all'euristica.
    """)
    
    print("="*70)
    print("  BENCHMARK A* COMPLETATO")
    print("="*70)
    
    return results


# ==============================================================================
# MAIN
# ==============================================================================

if __name__ == "__main__":
    # Configurazione benchmark
    # NOTA: A* può richiedere molta memoria per istanze grandi
    # Consigliato max_nodes <= 12 per tempi ragionevoli
    
    results = run_astar_benchmark(
        filepath="route_van_0.csv",
        max_nodes=10  # Limitato per evitare problemi di memoria
    )


  BENCHMARK: A* Algorithm per Traveling Salesman Problem
  Corso: Intelligenza Artificiale - Ricerca Informata

📂 Caricamento dati da: route_van_0.csv
   Limite nodi: 10 (+ deposito)
   Nodi caricati: 11 (1 deposito + 10 consegne)

📐 Pre-calcolo matrice distanze 11x11...

----------------------------------------------------------------------
🏃 ALGORITMO 1: Greedy Best-First Search (Baseline)
----------------------------------------------------------------------
   Percorso: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 0]
   Distanza: 45.55 km
   Tempo: 0.000254 sec

----------------------------------------------------------------------
🔬 ALGORITMO A* con Euristica: MST (Minimum Spanning Tree)
----------------------------------------------------------------------

🔍 Avvio A* con euristica: MST
   ✅ Soluzione trovata dopo 143 espansioni

   Percorso: [0, 1, 2, 6, 7, 5, 4, 3, 8, 9, 10, 0]
   Distanza: 45.42 km
   Tempo: 0.0049 sec
   Nodi espansi: 143
   Nodi generati: 684
   Max frontiera: 542

-